# Modèle Overfitting (sur-apprentissage)

Objectif : illustrer le **sur-apprentissage**. On entraîne volontairement un modèle qui **mémorise** les données d'entraînement (un seul arbre, sans bootstrap).

## 1. Pré-traitement (repris de `preprocessing.ipynb`)

In [1]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

In [2]:
data = pd.read_csv('data/kc_house_data.csv')
data = data.drop('id', axis=1)
data = data.drop('sqft_above', axis=1)  # colinéaire avec sqft_living

In [3]:
# Feature engineering : âge du bien + binarisation de la rénovation
data['date'] = pd.to_datetime(data['date'], format='%Y%m%dT%H%M%S')
data['yr_sold'] = data['date'].dt.year
data['house_age'] = data['yr_sold'] - data['yr_built']
data['was_renovated'] = (data['yr_renovated'] > 0).astype(int)
data = data.drop(['date', 'yr_built', 'yr_renovated'], axis=1)

In [4]:
# Split stratifié (cible condition très déséquilibrée)
X = data.drop('condition', axis=1)
y = data['condition']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train.shape, X_test.shape

((17290, 18), (4323, 18))

In [5]:
def evaluate(model):
    pred_train = model.predict(X_train)
    pred_test = model.predict(X_test)
    return pd.Series({
        'acc_train': accuracy_score(y_train, pred_train),
        'acc_test': accuracy_score(y_test, pred_test),
        'f1_train': f1_score(y_train, pred_train, average='macro'),
        'f1_test': f1_score(y_test, pred_test, average='macro'),
    }).round(3)

## 2. Modèle volontairement trop complexe

Un seul arbre (`n_estimators=1`), sans élagage et sans `bootstrap` : il apprend le train par coeur.

In [6]:
rf_overfit = RandomForestClassifier(
    n_estimators=1, bootstrap=False, random_state=42
)
rf_overfit.fit(X_train, y_train)
evaluate(rf_overfit)

acc_train    1.000
acc_test     0.630
f1_train     1.000
f1_test      0.305
dtype: float64

## 3. Diagnostic

**Signature du sur-apprentissage : `acc_train` proche de 1 mais `f1_test` bien plus bas.**
Le modèle a mémorisé les exemples d'entraînement (variance élevée) et généralise mal sur des données nouvelles. La solution : régulariser (limiter la profondeur, augmenter `min_samples_leaf`, moyenner plusieurs arbres via `n_estimators` + `bootstrap`), ce qui est fait dans le modèle final de `preprocessing.ipynb`.